# ARAD (Autoencoder Reconstruction Anomaly Detection)

This notebook demonstrates the ARAD algorithm for identifying radioactive
sources in gamma-ray time series data using a trained convolutional
autoencoder.

## Algorithm Overview

The ARAD detector:
1. **Learns background patterns** using a convolutional autoencoder
2. **Computes reconstruction error** (JSD or Chi-squared) between input
   and reconstruction
3. **Detects anomalies** when reconstruction error exceeds threshold
4. **Aggregates alarms** that are close in time

## Reference

Ghawaly J *et al.* (2022). Characterization of the Autoencoder Radiation
Anomaly Detection (ARAD) model. *Engineering Applications of Artificial
Intelligence*, 111, 104761.

## Dataset

Using the TopCoder Urban Data Challenge dataset (mobile NaI detector).

**Note:** You must download the TopCoder dataset separately and set
`TOPCODER_DATA_DIR` below to point to it.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

from gammaflow import SpectralTimeSeries
from gammaflow.algorithms import ARADDetector
from gammaflow.datasets import TopCoderDataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

# Path to the TopCoder dataset on your machine
TOPCODER_DATA_DIR = '/PATH/TO/topcoder'

ds = TopCoderDataset(TOPCODER_DATA_DIR)
print(ds)

## Train ARAD on Background Data

We load background runs, convert them to time series, and train the
autoencoder.  If you have a pre-trained model saved to disk, skip this
cell and use `detector.load(path)` instead.

In [ ]:
background_run_ids = ds.list_runs(source_id=0)
print(f"Found {len(background_run_ids)} background runs")

# Build training time series from background runs
bg_spectra_list = []
for run_id in tqdm(background_run_ids, desc='Loading background'):
    try:
        listmode, _ = ds.load_run(run_id)
        ts = SpectralTimeSeries.from_list_mode(
            listmode,
            integration_time=3.0,
            stride_time=1.0,
            energy_bins=128,
            energy_range=(20, 2900),
        )
        bg_spectra_list.append(ts)
    except Exception as e:
        print(f"  Warning: run {run_id} failed: {e}")

# Concatenate all background spectra into one time series for training
all_bg_spectra = []
for ts in bg_spectra_list:
    all_bg_spectra.extend(ts.spectra)

training_ts = SpectralTimeSeries(all_bg_spectra)
print(f"Training set: {training_ts.n_spectra} spectra")

In [ ]:
detector = ARADDetector(
    latent_dim=8,
    dropout=0.2,
    batch_size=32,
    learning_rate=0.01,
    epochs=50,
    aggregation_gap=2.0,
    loss_type='chi2',
    verbose=True,
)

detector.fit(training_ts)

# Optionally save the trained model for later reuse
# detector.save('arad_background.pt')

## Set Threshold Based on False Alarm Rate

ANSI N42.48 typically requires < 1 alarm/hour.  We concatenate
background runs to build at least 1 hour of calibration data for
reliable FAR estimation.

In [ ]:
alarms_per_hour = 0.5
MIN_CALIBRATION_HOURS = 1.0

# Build a calibration dataset from multiple background runs (>= 1 hour)
cal_spectra = []
cal_total_time = 0.0
for ts in bg_spectra_list:
    cal_spectra.extend(ts.spectra)
    cal_total_time += sum(s.real_time for s in ts.spectra)
    if cal_total_time >= MIN_CALIBRATION_HOURS * 3600.0:
        break

cal_counts = np.array([s.counts for s in cal_spectra])
cal_real_times = np.array([s.real_time for s in cal_spectra])
cal_live_times = np.array([
    s.live_time if s.live_time is not None else s.real_time
    for s in cal_spectra
])
cal_timestamps = np.cumsum(cal_real_times)
cal_energy_edges = bg_spectra_list[0].spectra[0].energy_edges

cal_time_series = SpectralTimeSeries.from_array(
    cal_counts,
    energy_edges=cal_energy_edges,
    timestamps=cal_timestamps,
    real_times=cal_real_times,
    live_times=cal_live_times,
)

print(f"Calibration dataset: {len(cal_spectra)} spectra, "
      f"{cal_total_time/3600:.2f} hours")

threshold = detector.set_threshold_by_far(
    cal_time_series,
    alarms_per_hour=alarms_per_hour,
    verbose=True,
)
print(f"\nThreshold: {threshold:.4f}")

## Test on Run with Source

In [ ]:
# Pick the strongest I-131 run
ak = ds.get_answer_key('training')
i131_runs = ak[ak['SourceID'] == 3].sort_values('Speed/Offset', ascending=False)
test_run_id = int(i131_runs.iloc[0]['RunID'])

listmode, metadata = ds.load_run(test_run_id)

print(f"Run {test_run_id}")
print(f"  Source: {metadata['SourceName']}")
print(f"  Source Time: {metadata['SourceTime']:.1f} s")
print(f"  Speed/Offset: {metadata['Speed/Offset']:.2f}")

test_ts = SpectralTimeSeries.from_list_mode(
    listmode,
    integration_time=3.0,
    stride_time=1.0,
    energy_bins=128,
    energy_range=(20, 2900),
)
print(f"  {test_ts}")

## Run ARAD Detection

In [ ]:
arad_scores = detector.process_time_series(test_ts)
summary = detector.get_alarm_summary()

print(f"Alarms: {summary['n_alarms']}")
if summary['n_alarms'] > 0:
    print(f"Peak ARAD score: {summary['max_peak_metric']:.4f}")
    print(f"Total alarm time: {summary['total_alarm_time']:.2f} s")

true_source_time = metadata['SourceTime']
for i, alarm in enumerate(detector.alarms, 1):
    print(f"  {i}. {alarm}")
    if alarm.start_time <= true_source_time <= alarm.end_time:
        print(f"      Captured true source (t={true_source_time:.1f}s)")

## Visualize Results

In [ ]:
times = test_ts.timestamps
count_rates = np.array([
    float(s.counts.sum()) / float(
        s.live_time if s.live_time is not None and not np.isnan(s.live_time)
        else s.real_time
    )
    for s in test_ts.spectra
])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Count rate
ax1.step(times, count_rates, where='post', color='black', lw=1.5,
         label='Count rate')
ax1.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax1.set_title(
    f'ARAD Detection - Run {test_run_id} ({metadata["SourceName"]})',
    fontsize=14, fontweight='bold',
)
ax1.grid(True, alpha=0.3)
for i, alarm in enumerate(detector.alarms):
    ax1.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if i == 0 else '')
if metadata['SourceID'] != 0:
    ax1.axvline(metadata['SourceTime'], color='green', ls='--', lw=2,
                label='True source')
ax1.legend(loc='best')

# ARAD scores
ax2.step(times, arad_scores, where='post', color='steelblue', lw=1.5,
         label='ARAD Score')
ax2.axhline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({detector.threshold:.4f})')
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('ARAD Score', fontsize=12)
ax2.set_title('ARAD Score Time Series', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
for alarm in detector.alarms:
    ax2.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')
ax2.legend(loc='best')

plt.tight_layout()
plt.show()

## Explainability: Saliency Maps

ARAD includes explainable AI capabilities through saliency maps.
These show which energy bins contribute most to the anomaly score,
helping us understand *why* the model flagged a spectrum.

In [ ]:
if detector.alarms:
    peak_alarm = max(detector.alarms, key=lambda a: a.peak_metric)
    peak_idx = np.argmin(np.abs(times - peak_alarm.peak_time))
    peak_spectrum = test_ts[peak_idx]

    print(f"Analyzing peak spectrum at t={times[peak_idx]:.1f}s "
          f"(score={arad_scores[peak_idx]:.4f})")

    fig, axes = detector.plot_saliency(
        peak_spectrum,
        method='gradient',
        figsize=(14, 10),
        show_reconstruction=True,
    )
    plt.show()
else:
    print("No alarms detected. Try lowering the threshold.")